In [1]:
pip install transformers torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/DialoGPT-medium"

print("Loading model... Please wait.")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

chat_history_ids = None

print("\nChatbot: Hello! I am your AI assistant. How can I help you today?")

while True:
    user_input = input("User: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a nice day 😊")
        break

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # ✅ FIX: Create attention mask
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,   # ✅ IMPORTANT FIX
        max_length=200,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=100,
        top_p=0.95,
        temperature=0.8,
        no_repeat_ngram_size=3
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    if response.strip() == "":
        response = "I'm not sure how to respond. Can you rephrase?"

    print("Chatbot:", response)

Loading model... Please wait.


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Chatbot: Hello! I am your AI assistant. How can I help you today?
User: hello
Chatbot: what up
User: what is AI
Chatbot: lol what
